In [1]:
!nvidia-smi

Mon Apr 27 06:55:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   40C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip -q install -U transformers datasets evaluate accelerate huggingface_hub sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 152.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.3 MB/s eta 0:00:00


In [3]:
import os, gc, json, math, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from huggingface_hub import snapshot_download
from transformers import (
    AutoTokenizer,                   # Tự động chọn tokenizer đúng cho model
    AutoModelForQuestionAnswering,   # Model QA: output start/end logits
    TrainingArguments,               # Hyperparameter cho huấn luyện
    Trainer,                         # Vòng lặp train/eval tiêu chuẩn
    default_data_collator,           # Gom batch mặc định
)

In [4]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)   # Seed cho tất cả GPU

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [5]:
CONFIG = {
    "MODEL_NAME": "valhalla/longformer-base-4096-finetuned-squadv1",
    "MAX_LENGTH": 2048,        # Độ dài tối đa mỗi cửa sổ (tokens)
    "DOC_STRIDE": 256,         # Overlap giữa các cửa sổ
    "TRAIN_CONTRACTS": 80,     # Số hợp đồng cho train
    "VAL_CONTRACTS": 12,       # Số hợp đồng cho validation
    "NUM_EPOCHS": 1,           # 1 epoch = đủ xem pipeline, tăng lên 3+ nếu muốn tốt hơn
    "MAX_TRAIN_QA": 2000,      # Giới hạn số QA pairs dùng để train
    "MAX_VAL_QA": 200,         # Giới hạn số QA pairs để validate
    "TRAIN_BATCH_SIZE": 2,     # Nhỏ vì mỗi sample dài 2048 token
    "EVAL_BATCH_SIZE": 2,
    "LR": 2e-5,                  # Learning rate chuẩn cho fine-tune Transformer
    "GRAD_ACCUM": 4,            # Effective batch = 2 * 4 = 8
    "MAX_ANSWER_TOKENS": 40,   # Câu trả lời pháp lý thường ngắn
}

In [6]:
cuad_root = snapshot_download(
    repo_id="theatticusproject/cuad",
    repo_type="dataset",
    allow_patterns=[
        "CUAD_v1/CUAD_v1.json",             # File QA dạng SQuAD
        "CUAD_v1/full_contract_txt/**/*.txt", # Toàn văn hợp đồng TXT
    ],
    local_dir="./hf_cuad",
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 201 files:   0%|          | 0/201 [00:00<?, ?it/s]

In [7]:
cuad_root = Path(cuad_root)
json_path = cuad_root / "CUAD_v1" / "CUAD_v1.json"
txt_root  = cuad_root / "CUAD_v1" / "full_contract_txt"

print("json exists:", json_path.exists())
print("txt root exists:", txt_root.exists())

json exists: True
txt root exists: True


In [8]:
def flatten_cuad_squad(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    rows = []
    for article in raw["data"]:
        title = article["title"].strip()
        for paragraph in article["paragraphs"]:
            context = paragraph["context"].strip()
            for qa in paragraph["qas"]:
                answers = qa.get("answers", [])
                answer_texts  = [a["text"].strip() for a in answers]
                answer_starts = [int(a["answer_start"]) for a in answers]

                rows.append({
                    "id": qa["id"],
                    "title": title,
                    "context": context,
                    "question": qa["question"].strip(),
                    "answers": {
                        "text": answer_texts,
                        "answer_start": answer_starts,
                    },
                    "is_impossible": len(answer_texts) == 0, # Không có đáp án
                    "context_chars": len(context),
                })
    return rows

In [9]:
all_rows = flatten_cuad_squad(json_path)
len(all_rows)  # → ~13,000 QA examples

20910

In [10]:
txt_files = list(txt_root.glob("**/*.txt"))
title_to_txt = {}
for p in txt_files:
    title_to_txt[p.stem] = p  # stem = tên file không có extension

# Chỉ giữ QA rows có file txt tương ứng
all_rows = [r for r in all_rows if r["title"] in title_to_txt]
print("QA sau khi khớp với full txt:", len(all_rows))

QA sau khi khớp với full txt: 8118


In [26]:
titles = sorted({r["title"] for r in all_rows})
random.Random(SEED).shuffle(titles)   # Shuffle với seed cố định

train_titles = titles[:CONFIG["TRAIN_CONTRACTS"]]   # 80 hợp đồng đầu
val_titles   = titles[80:80+CONFIG["VAL_CONTRACTS"]]  # 12 hợp đồng tiếp

train_rows = [r for r in all_rows if r["title"] in train_titles]
val_rows   = [r for r in all_rows if r["title"] in val_titles]

# Giới hạn số lượng để chạy nhanh trên Colab
if len(train_rows) > CONFIG["MAX_TRAIN_QA"]:
    train_rows = random.Random(SEED).sample(train_rows, CONFIG["MAX_TRAIN_QA"])

raw_datasets = DatasetDict({
    "train": Dataset.from_list(train_rows),
    "validation": Dataset.from_list(val_rows),
})

# Lọc ra các val rows có đáp án thật (is_impossible=False) → dùng ở bước demo inference
answerable_val_rows = [r for r in val_rows if not r["is_impossible"]]
print(f"Val rows có đáp án: {len(answerable_val_rows)} / {len(val_rows)}")

Val rows có đáp án: 135 / 492


In [12]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["MODEL_NAME"])
model = AutoModelForQuestionAnswering.from_pretrained(CONFIG["MODEL_NAME"])

model.config.use_cache = False   # Tắt key-value cache khi train
model.to(device)                   # Chuyển sang GPU

print("Tokenizer max length:", tokenizer.model_max_length)  # → 4096
print("CLS token id:", tokenizer.cls_token_id)               # → 0

config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/595M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/271 [00:00<?, ?it/s]

[transformers] LongformerForQuestionAnswering LOAD REPORT from: valhalla/longformer-base-4096-finetuned-squadv1
Key                            | Status     |  | 
-------------------------------+------------+--+-
longformer.pooler.dense.weight | UNEXPECTED |  | 
longformer.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenizer max length: 4096
CLS token id: 0


In [13]:
max_length = CONFIG["MAX_LENGTH"]   # 2048
doc_stride = CONFIG["DOC_STRIDE"]   # 256

In [14]:
def prepare_train_features(examples):
    questions = [q.lstrip() for q in examples["question"]]

    tokenized_examples = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",       # Chỉ cắt context, không cắt câu hỏi
        stride=doc_stride,               # Overlap giữa các cửa sổ
        return_overflowing_tokens=True,  # Trả về TẤT CẢ cửa sổ
        return_offsets_mapping=True,    # Ánh xạ token → ký tự gốc
        padding="max_length",
    )

    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")

    start_positions = []
    end_positions   = []

    for i, offsets in enumerate(offset_mapping):
        input_ids    = tokenized_examples["input_ids"][i]
        cls_index    = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized_examples.sequence_ids(i)
        # sequence_id=0 → token thuộc câu hỏi
        # sequence_id=1 → token thuộc context

        sample_idx = sample_mapping[i]
        answers    = examples["answers"][sample_idx]

        # Câu hỏi không có đáp án → CLS
        if len(answers["answer_start"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_char = answers["answer_start"][0]
        end_char   = start_char + len(answers["text"][0])

        # Tìm token đầu/cuối của phần context trong cửa sổ
        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1
        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        # Đáp án ngoài cửa sổ này → CLS
        if (offsets[token_start_index][0] > start_char
                or offsets[token_end_index][1] < end_char):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            # Tìm token bắt đầu chính xác của đáp án
            while (token_start_index < len(offsets)
                   and offsets[token_start_index][0] <= start_char):
                token_start_index += 1
            start_positions.append(token_start_index - 1)

            while offsets[token_end_index][1] >= end_char:
                token_end_index -= 1
            end_positions.append(token_end_index + 1)

    tokenized_examples["start_positions"] = start_positions
    tokenized_examples["end_positions"]   = end_positions
    return tokenized_examples

In [15]:
tokenized_datasets = raw_datasets.map(
    prepare_train_features,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,  # Xóa cột gốc
)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/492 [00:00<?, ? examples/s]

In [16]:
tokenized_sample = tokenized_datasets["train"][0]
for k, v in tokenized_sample.items():
    if isinstance(v, list):
        print(k, len(v))
    else:
        print(k, type(v))

input_ids 2048
attention_mask 2048
start_positions <class 'int'>
end_positions <class 'int'>


In [17]:
training_args = TrainingArguments(
    output_dir="./longformer-cuad-checkpoints",
    learning_rate=CONFIG["LR"],            # 2e-5
    num_train_epochs=CONFIG["NUM_EPOCHS"], # 1
    per_device_train_batch_size=CONFIG["TRAIN_BATCH_SIZE"],  # 2
    per_device_eval_batch_size=CONFIG["EVAL_BATCH_SIZE"],
    gradient_accumulation_steps=CONFIG["GRAD_ACCUM"],  # 4
    eval_strategy="epoch",    # Eval sau mỗi epoch
    save_strategy="no",       # Không lưu checkpoint giữa chừng
    logging_steps=25,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),   # FP16 nếu có GPU → nhanh ~2x, ít VRAM hơn
    report_to="none",
)

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=default_data_collator,
)

trainer.train()   # 🏋️ Bắt đầu huấn luyện

Epoch,Training Loss,Validation Loss
1,0.355321,0.279649


TrainOutput(global_step=1814, training_loss=0.3504263203356994, metrics={'train_runtime': 4985.5566, 'train_samples_per_second': 2.91, 'train_steps_per_second': 0.364, 'total_flos': 1.89513060937728e+16, 'train_loss': 0.3504263203356994, 'epoch': 1.0})

In [19]:
save_dir = "./longformer-cuad-model"
trainer.save_model(save_dir)         # Lưu weights + config
tokenizer.save_pretrained(save_dir)  # Lưu vocab + tokenizer config
print("Đã lưu model tại:", save_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu model tại: ./longformer-cuad-model


In [23]:
def answer_long_document(question, context, model, tokenizer,
                           max_length=2048, doc_stride=256, max_answer_tokens=40):
    model.eval()  # Tắt dropout, BatchNorm, v.v.

    # Tokenize → nhiều cửa sổ nếu hợp đồng quá dài
    encoded = tokenizer(
        [question], [context],
        max_length=max_length, truncation="only_second", stride=doc_stride,
        return_overflowing_tokens=True, return_offsets_mapping=True,
        padding="max_length",
    )

    best_answer = {"text": "", "score": -1e9, "start_char": None, "end_char": None}

    for i in range(len(encoded["input_ids"])):
        input_ids     = torch.tensor([encoded["input_ids"][i]], device=device)
        attention_mask = torch.tensor([encoded["attention_mask"][i]], device=device)

        with torch.no_grad():   # Không tính gradient → tiết kiệm bộ nhớ & nhanh hơn
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        start_logits = outputs.start_logits[0].detach().cpu().numpy()
        end_logits   = outputs.end_logits[0].detach().cpu().numpy()
        offsets      = encoded["offset_mapping"][i]
        sequence_ids = encoded.sequence_ids(i)

        # Top-20 candidates cho cả start và end
        start_indexes = np.argsort(start_logits)[-20:][::-1]
        end_indexes   = np.argsort(end_logits)[-20:][::-1]

        for s_idx in start_indexes:
            for e_idx in end_indexes:
                # Validate: cả hai phải nằm trong context (seq_id=1)
                if sequence_ids[s_idx] != 1 or sequence_ids[e_idx] != 1: continue
                if e_idx < s_idx: continue  # End phải sau Start
                if (e_idx - s_idx + 1) > max_answer_tokens: continue  # Không quá dài

                start_char = offsets[s_idx][0]
                end_char   = offsets[e_idx][1]
                pred_text  = context[start_char:end_char].strip()
                if not pred_text: continue

                score = float(start_logits[s_idx] + end_logits[e_idx])
                if score > best_answer["score"]:
                    best_answer = {
                        "text": pred_text, "score": score,
                        "start_char": start_char, "end_char": end_char,
                        "feature_index": i,
                    }

    return best_answer

In [28]:
# Lấy một QA example từ validation set
demo_row         = answerable_val_rows[0]
demo_question    = demo_row["question"]
demo_gold_answer = demo_row["answers"]["text"][0]
full_contract    = Path(title_to_txt[demo_row["title"]]).read_text(encoding="utf-8", errors="ignore")
full_contract_text = full_contract  # Alias dùng ở Bước 13 — custom question

pred = answer_long_document(
    question=demo_question,
    context=full_contract,
    model=model, tokenizer=tokenizer,
    max_length=CONFIG["MAX_LENGTH"],
    doc_stride=CONFIG["DOC_STRIDE"],
    max_answer_tokens=CONFIG["MAX_ANSWER_TOKENS"],
)

print("QUESTION:",   demo_question)
print("PREDICTED:",  pred["text"])
print("GOLD:",       demo_gold_answer)

QUESTION: Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract
PREDICTED: STRATEGIC ALLIANCE AGREEMENT
GOLD: STRATEGIC ALLIANCE AGREEMENT


In [29]:
custom_question = "Highlight the parts (if any) of this contract related to \
'Governing Law' that should be reviewed by a lawyer. \
Details: Which law governs this contract?"

custom_pred = answer_long_document(
    question=custom_question,
    context=full_contract_text,   # Toàn bộ hợp đồng thật
    model=model,
    tokenizer=tokenizer,
    max_length=CONFIG["MAX_LENGTH"],
    doc_stride=CONFIG["DOC_STRIDE"],
    max_answer_tokens=CONFIG["MAX_ANSWER_TOKENS"],
)

print("QUESTION:", custom_question)
print("ANSWER:",   custom_pred["text"])
print("SCORE:",    custom_pred["score"])

QUESTION: Highlight the parts (if any) of this contract related to 'Governing Law' that should be reviewed by a lawyer. Details: Which law governs this contract?
ANSWER: The construction, interpretation, and performance of this Agreement and all transactions under it shall be governed by the laws of the State of Texas, irrespective of its conflict of law principles.
SCORE: 20.8984375
